# FHIR Patient Pipeline - Exploration Notebook

This notebook walks through the full pipeline output, demonstrating:
- Reading FHIR bundles from `data/raw/`
- Validating resources and reviewing quality scores
- Inspecting Bronze, Silver, and Gold layer outputs
- Running analytical queries against the DuckDB warehouse

Run the pipeline first: `python -m src.pipeline.orchestrator`

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path
import pandas as pd
import duckdb

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
print('Dependencies loaded.')

## 1. Run the Pipeline

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(name)s | %(message)s')

from src.pipeline.orchestrator import PipelineOrchestrator

orchestrator = PipelineOrchestrator(config_path='../config/pipeline_config.yaml')
summary = orchestrator.run()

print(f"\nRun ID  : {summary['run_id']}")
print(f"Status  : {summary['status']}")
print(f"Duration: {summary['duration_seconds']:.2f}s")
print(f"Records : {summary['total_records_processed']}")

In [ ]:
# Review per-stage results
stages_df = pd.DataFrame(summary['stages'])[['stage', 'status', 'records_processed', 'duration_seconds']]
stages_df

## 2. Inspect FHIR Bundle Contents

In [ ]:
from src.ingestion.fhir_bundle_reader import FHIRBundleReader

reader = FHIRBundleReader()
resources, meta_list = reader.read_directory('../data/raw')

summary_info = reader.get_summary(resources)
print('Resource counts:')
for rt, count in summary_info['resource_counts'].items():
    print(f'  {rt:15s}: {count}')

In [ ]:
# Look at one patient
patient = resources['Patient'][0]
print('Patient ID :', patient['id'])
print('Name       :', patient.get('name', [{}])[0].get('family'))
print('DOB        :', patient.get('birthDate'))
print('Gender     :', patient.get('gender'))

## 3. Validation Results

In [ ]:
from src.validation.fhir_validator import FHIRValidator
from src.validation.data_quality_checks import DataQualityEngine

validator = FHIRValidator()
quality_engine = DataQualityEngine(min_quality_score=70)

for resource_type, resource_list in resources.items():
    if not resource_list:
        continue
    results = validator.validate_batch(resource_list)
    valid_count = sum(1 for r in results if r.is_valid)
    
    report = quality_engine.run_checks(resource_list, resource_type)
    print(f'{resource_type:15s}: {valid_count}/{len(results)} valid | Quality score: {report.quality_score:.1f} | Passed: {report.passed_pipeline_threshold}')

## 4. Bronze Layer

In [ ]:
from src.transformation.bronze_layer import BronzeLayerProcessor

bronze_proc = BronzeLayerProcessor(bronze_zone_path='../data/bronze')
bronze_patients = bronze_proc.read_bronze('Patient')

if not bronze_patients.empty:
    print(f'Bronze patients: {len(bronze_patients)} rows')
    print('Columns:', list(bronze_patients.columns))
    bronze_patients[['resource_id', 'version_id', 'source_file', 'ingestion_date', 'pipeline_run_id']].head()

## 5. Silver Layer

In [ ]:
from src.transformation.silver_layer import SilverLayerProcessor

silver_proc = SilverLayerProcessor(silver_zone_path='../data/silver')

patients_df = silver_proc.read_silver('Patient')
print('Silver patients:')
patients_df[['patient_id', 'full_name', 'gender', 'birth_date', 'city', 'state']]

In [ ]:
encounters_df = silver_proc.read_silver('Encounter')
print('Silver encounters:')
encounters_df[['encounter_id', 'patient_id', 'encounter_class', 'status', 'period_start', 'period_end', 'length_of_stay_hours']]

In [ ]:
conditions_df = silver_proc.read_silver('Condition')
print('Silver conditions (active only):')
active = conditions_df[conditions_df['is_active'] == True]
active[['patient_id', 'condition_code', 'condition_display', 'onset_date', 'clinical_status']]

In [ ]:
obs_df = silver_proc.read_silver('Observation')
print('Silver observations (vitals):')
vitals = obs_df[obs_df['category'] == 'vital-signs']
vitals[['patient_id', 'observation_name', 'value', 'unit', 'effective_date']].head(10)

## 6. Gold Layer

In [ ]:
from src.transformation.gold_layer import GoldLayerProcessor

gold_proc = GoldLayerProcessor(gold_zone_path='../data/gold')

enc_summary = gold_proc.read_gold('patient_encounter_summary')
print('Patient Encounter Summary:')
cols = ['patient_id', 'full_name', 'total_encounters', 'amb_encounters', 'imp_encounters', 'emer_encounters', 'avg_los_hours']
enc_summary[[c for c in cols if c in enc_summary.columns]]

In [ ]:
condition_profile = gold_proc.read_gold('patient_condition_profile')
print('Patient Condition Profile:')
cols = ['patient_id', 'full_name', 'active_condition_count', 'chronic_condition_count', 'has_chronic_condition', 'active_icd_codes']
condition_profile[[c for c in cols if c in condition_profile.columns]]

In [ ]:
financial_summary = gold_proc.read_gold('claims_financial_summary')
print('Claims Financial Summary:')
cols = ['patient_id', 'full_name', 'total_claims', 'total_billed', 'avg_claim_value', 'professional_claims', 'institutional_claims']
financial_summary[[c for c in cols if c in financial_summary.columns]]

In [ ]:
trends = gold_proc.read_gold('monthly_utilization_trends')
print('Monthly Utilization Trends:')
trends[['year_month', 'encounter_class', 'status', 'encounter_count', 'avg_los_hours']].sort_values('year_month')

## 7. DuckDB Warehouse Queries

In [ ]:
db_path = '../data/warehouse/fhir_warehouse.duckdb'

if Path(db_path).exists():
    con = duckdb.connect(db_path, read_only=True)
    
    print('Tables in warehouse:')
    tables = con.execute("""
        SELECT table_schema, table_name, table_type
        FROM information_schema.tables
        WHERE table_schema IN ('bronze', 'silver', 'gold')
        ORDER BY table_schema, table_name
    """).df()
    print(tables.to_string(index=False))
else:
    print(f'Warehouse not found at {db_path}. Run the pipeline first.')

In [ ]:
# Patient 360 view
if Path(db_path).exists():
    patient_360 = con.execute("""
        SELECT
            patient_id,
            full_name,
            gender,
            total_encounters,
            chronic_condition_count,
            total_billed,
            risk_tier,
            cost_per_encounter
        FROM gold.vw_patient_360
        ORDER BY total_billed DESC
    """).df()
    print('Patient 360 View:')
    print(patient_360.to_string(index=False))

In [ ]:
# Chronic disease burden
if Path(db_path).exists():
    burden = con.execute("""
        SELECT *
        FROM gold.vw_chronic_disease_burden
        ORDER BY total_patient_occurrences DESC
    """).df()
    print('Chronic Disease Burden:')
    print(burden.to_string(index=False))

In [ ]:
# High-cost patients
if Path(db_path).exists():
    high_cost = con.execute("""
        SELECT
            patient_id,
            full_name,
            total_billed,
            chronic_condition_count,
            cost_tier
        FROM gold.vw_high_cost_patients
    """).df()
    print('High Cost Patients:')
    print(high_cost.to_string(index=False))
    con.close()

## 8. Incremental Load Example

In [ ]:
from src.pipeline.incremental_load import IncrementalLoadManager

manager = IncrementalLoadManager('../data/watermarks.json')

print('Current watermarks:')
for source, info in manager.list_sources().items():
    print(f"  {source}: last processed = {info.get('last_processed_timestamp', 'never')}")
    print(f"           files in last run = {info.get('files_processed_in_last_run', 0)}")

In [ ]:
# Simulate checking for new files
pending = manager.get_pending_files('../data/raw', source_key='fhir_bundles')
print(f'Pending files for incremental load: {len(pending)}')
for f in pending:
    print(f'  - {f.name}')